# 03 · Comparison (SQ1 + SQ2)
**Owner:** _your name_ · Group 20 · Dutch Housing Blame

**Research question.** Who do people blame for the Dutch housing crisis online, and do Dutch-language
(r/thenetherlands) and English-language (r/Netherlands) communities blame different actors?

**Goal of this notebook.** Answer SQ1 (which actors does each community blame most?) and SQ2 (where do they agree and differ?).

**Input:** `annotation/labels_final.csv` (from 02), `data/processed/comments_housing.parquet` (from 01)
**Output:** the main result figures and tables of the paper.

**Done when:**
- [ ] every figure/table for the paper is produced here
- [ ] every test is explained in markdown
- [ ] runs on the real labels

**How to work:** each task has a few parts. A part says what has to exist when you're done, not how to build it –
split it into as many cells and functions as you like. Names in `code font` are used by the check cells or by other
notebooks, so keep them. *Go further* items are optional, for when we need more depth.

Read `notebooks/GUIDE.md` first (rules + the column names everyone uses).

## Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 20
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
ANNOTATION = ROOT / "annotation"
FIGURES = ROOT / "figures"
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

## Task 0 · Fake labels
**Why (for the paper):** Lets you build and test everything now instead of waiting for the annotation.

**0A. Generator**
- `make_fake_labels(df, n=300)`: random labels with the columns of `labels_final.csv`.
- A built-in difference between the subreddits, so you can see later whether your tests find it.

In [ ]:
# check (do not change)
fake = make_fake_labels(pd.DataFrame({"id": [f"x{i}" for i in range(500)],
                                       "subreddit": ["thenetherlands", "Netherlands"] * 250}), n=300)
assert len(fake) == 300 and any(c.startswith("target_") for c in fake.columns)
print("ok")

## Task 1 · SQ1: who is blamed most?
**Why (for the paper):** The main answer to SQ1.

**1A. Base**
- `base`: labelled comments merged with their metadata, only relevant comments with an attribution (agree with 02 on `unclear`).
- The size of the base per subreddit and language – this is the denominator of every percentage.

**1B. Shares**
- `sq1`: per subreddit and target the share of comments that *blame* the target, with a 95% Wilson interval (`share`, `ci_low`, `ci_high`).
- A grouped bar chart with error bars in `figures/03_sq1_shares.png` (shares don't add up to 100% – targets are multi-label).

In [ ]:
# check (do not change)
assert {"subreddit", "target", "share", "ci_low", "ci_high"}.issubset(sq1.columns)
assert ((sq1["ci_low"] <= sq1["share"]) & (sq1["share"] <= sq1["ci_high"])).all()
print("ok")

**Go further (optional):**
- The same split by attribution type (causal vs. moral).

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Task 2 · SQ2: where do they differ and agree?
**Why (for the paper):** Turns "looks different" into a statement we can defend, plus one number for overall agreement.

**2A. Differences**
- `sq2`: per target a test of the difference between the subreddits, the difference in percentage points, `p_raw` and `p_corrected` (corrected for testing many targets).
- A short explanation of why you chose this test and this correction.

**2B. Overall similarity**
- One similarity number between the two blame profiles (cosine and/or Jaccard, as in A4), with a bootstrap interval.
- A baseline that makes the number meaningful (e.g. two random halves of one community).

**2C. Blames vs. defends**
- Per target and community: blamed, defended, accusation disputed – as a stacked bar chart in `figures/03_positions.png`.

In [ ]:
# check (do not change)
assert (sq2["p_corrected"] >= sq2["p_raw"] - 1e-12).all()
print(sq2.round(3))

**Go further (optional):**
- Odds ratios; a logistic regression with subreddit *and* language.
- Similarity per year: growing apart or together?

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Task 3 · Context: upvotes and time
**Why (for the paper):** Shows which blame each community *rewards*, and whether the debate follows events.

**3A. Upvotes**
- Which targets get more upvotes within each community? Scores depend on thread size, so compare fairly (e.g. rank within thread) and pick a suitable test.

**3B. Over time**
- Monthly housing comments per subreddit with 4–6 major events marked (find the dates and a source for each), in `figures/03_time.png`.

**Go further (optional):**
- For the biggest peaks: what were the top housing posts that week?

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Task 4 · Robustness
**Why (for the paper):** Our communities share many users, so subreddit ≠ language group. If both give the same picture, the result is stronger.

**4A. Language instead of subreddit**
- SQ1 again, grouped by `language` (nl / en), and a comparison with the subreddit result (e.g. rank correlation).

**Go further (optional):**
- Inside r/Netherlands only: Dutch- vs. English-language comments.

**Interpretation:** _2–4 sentences: what do you see, and why did you do it this way?_

## Save outputs
- All final figures as `figures/03_*.png`, the tables (`sq1`, `sq2`, similarity) as CSV in `data/processed/`.

In [ ]:
# check (do not change)
import glob
figs = glob.glob(str(FIGURES / "03_*.png"))
assert len(figs) >= 3, figs
print(len(figs), "figures saved")

## Notes for the paper
- Main figure: blame shares with CIs per community.
- Table: differences, corrected p-values, effect sizes.
- Similarity number and how to read it.